# Data pipeline: train / validation / test datasets

Builds training examples and labels for the FCN3 → hourly-wind refinement model (GitHub issue #1).
See `data/data.md` for a description of the raw data and the reasoning behind the split.

**Task (one *window*):** given ERA5 at `t0` and the FCN3 ensemble at `t0+6h`, predict ERA5 u/v at `t0+1h … t0+6h`.

The code lives in `src/balloon/data.py` (install with `pip install -e .`); this notebook runs and explains it.

The pipeline has two stages:

1. **Extract windows** (`extract_all`): slow, run once. For every month in `data/fcn3_region/`, it pulls out the
   pieces of each init that a phase-1 window needs and saves them, full-domain, to
   `data/processed/windows/YYYY-MM.nc`. It doesn't depend on the split, and months already extracted are skipped.
2. **Make examples** (`make_patch_dataset`): fast, run as often as you like. It assigns each window to
   train/val/test using `SPLIT_RULES`, then cuts `window_size × window_size` patches into arrays `X`, `Y`.
   `window_size=1` is the pointwise model.

**Adding data later:** drop new `fcn3_YYYY-MM.nc` + `era5_hourly_YYYY-MM.nc` pairs (same format) into
`data/fcn3_region/` and re-run the notebook. New months are extracted and assigned to splits by `SPLIT_RULES`.
Normalization stats are recomputed automatically when the training set changes.

In [1]:
from balloon.data import (REPO, HOURS, SPLIT_RULES, discover_months, extract_all, window_index,
                          load_stats, make_patch_dataset, denormalize_y)
import numpy as np

REPO

PosixPath('/home/hocke/Balloon_Trajectory_Prediction')

## 1. Discover raw months
A month is usable when both its FCN3 file and its hourly-ERA5 file exist.

In [2]:
MONTHS = discover_months()
MONTHS

['2020-01', '2020-07', '2021-01', '2021-07']

## 2. Split rules
Splits are assigned **per window by its init time**, never per patch, so every patch from one window lands in the
same split. Rules (team decision, see `data/data.md`):

- **test:** any init in `test_years`. These must be after FCN3's training period (1980–2016).
- **train only:** inits inside FCN3's training period. Training there is fine, since we learn finer time
  resolution, but validation should look like test.
- **val:** inits on `val_days` of every other month. **buffer:** `buffer_days` are dropped, so no truth hour is
  shared between train and val.
- **train:** everything else.

When more data is added, edit `test_years` in `SPLIT_RULES` (`src/balloon/data.py`) and nothing else.

In [3]:
SPLIT_RULES

{'test_years': [2021],
 'val_days': [26, 27, 28, 29, 30, 31],
 'buffer_days': [24, 25]}

## 3. Extract windows (stage 1, cached)
Each saved window contains, on the full 201×241 grid:

| variable | dims | source |
|---|---|---|
| `t0_uv` | (init, var=u/v, level=10, lat, lon) | ERA5 at t0, all 10 levels. This is real input: FCN3 starts from ERA5, so it is known at forecast time |
| `t0_zt` | (init, var=z/t, level=5, lat, lon) | FCN3 lead 0 (= ERA5) z, t |
| `t6` | (init, member, var=u/v/z/t, level=5, lat, lon) | FCN3 ensemble at lead 6h |
| `y` | (init, hour=1..6, var=u/v, level=10, lat, lon) | ERA5 truth at t0+1h … t0+6h |

Inits whose truth hours are missing from the ERA5 file are skipped. Data is written one init at a time, so memory
use stays around 50 MB.

In [4]:
%%time
extract_all()

CPU times: user 192 μs, sys: 18 μs, total: 210 μs
Wall time: 214 μs


[PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2020-01.nc'),
 PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2020-07.nc'),
 PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2021-01.nc'),
 PosixPath('/home/hocke/Balloon_Trajectory_Prediction/data/processed/windows/2021-07.nc')]

### Split summary

In [5]:
index = window_index()
index.fillna({"split": "buffer"}).pivot_table(index="month", columns="split", values="init",
                                              aggfunc="count", fill_value=0, margins=True)

split,buffer,test,train,val,All
month,,,,,
2020-01,4,0,46,12,62
2020-07,4,0,46,12,62
2021-01,0,62,0,0,62
2021-07,0,62,0,0,62
All,8,124,92,24,248


## 4. Normalization statistics (train split only)
Each physical variable/level (`u50`, `z100`, …) gets one mean/std pair, computed from the **train** windows' t0
fields. The same pair is applied to t0, the t6 members and the targets, so every field shares one scale. That keeps
residual-style models simple: target minus input stays meaningful.

Stats are cached in `stats.nc` together with a hash of the train inits, and recomputed when the training set changes.

In [6]:
stats = load_stats(index)
stats.to_dataframe().round(2).T

field,u30,u50,u70,u100,u125,u150,u175,u200,u225,u250,...,z50,z100,z150,z200,z250,t50,t100,t150,t200,t250
mean,-1.65,-3.01,0.52,6.83,12.44,14.81,15.33,14.79,13.78,12.66,...,202295.56,162277.51,138655.14,120988.15,106599.57,207.26,198.41,208.76,219.56,229.97
std,12.42,12.38,12.00,14.07,15.38,17.78,19.50,20.33,20.26,19.59,...,679.48,1257.61,1792.33,1901.62,1780.64,3.56,6.78,4.24,2.69,3.77


## 5. Make examples (stage 2)

`make_patch_dataset(split, window_size, ...)` returns a dict:

| key | shape | meaning |
|---|---|---|
| `X` | (N, C, window_size, window_size) | input channels, listed in `x_names` |
| `Y` | (N, 6, 2, L, label_size, label_size) | targets: hour 1–6 × (u, v) × `target_levels` |
| `meta` | DataFrame, N rows | `month, init, split, i, j`: the window's top-left grid index, for stitching/analysis |
| `x_names`, `target_levels`, `label_offset` | | bookkeeping |

**Input channels** (`C` = 20 + 10 + t6 + 3):
- `t0_u{lev}`, `t0_v{lev}`: ERA5 t0 at all 10 levels (20)
- `t0_z{lev}`, `t0_t{lev}`: t0 at the 5 FCN3 levels (10)
- t6, depending on `member_mode`:
  - `"stack"`: every member as its own channels, `t6_m{k}_{var}{lev}` (4×20=80). The model sees member order,
    which is arbitrary.
  - `"meanstd"`: `t6_mean_*`, `t6_std_*` (40). Permutation-invariant, and works if member counts differ between
    months.
  - `"mean"`: ensemble mean only (20)
- `sin_lat`, `sin_lon`, `cos_lon`: absolute position, since a patch doesn't otherwise know where it is (3)

**Choosing patches:**
- `window_size`: input patch side. **1 = pointwise.**
- `label_size`: the side of the central region that gets labels (default = `window_size`). Setting it smaller
  gives a context halo. For example, `window_size=64, label_size=48` gives an 8-cell border of input-only context.
  `window_size − label_size` must be even.
- Pass **either** `n_per_window` (random positions, for training; seeded) **or** `stride` (a regular grid, for
  evaluation). With `stride=label_size` the label regions tile the domain without gaps. The last row/column is
  shifted to touch the edge, and with a halo the outer `(window_size−label_size)/2` cells of the domain are never
  labelled.

Normalized values: `x_norm = (x − mean) / std` per physical field. `t6_std_*` channels are divided by std only.
Use `denormalize_y` to get m/s back.

## 6. Examples

In [7]:
# Pointwise (window_size=1): 2,000 random grid points per training window
point = make_patch_dataset("train", window_size=1, n_per_window=2000)
X_point = point["X"][:, :, 0, 0]          # (N, C) for an MLP
Y_point = point["Y"][..., 0, 0]           # (N, 6, 2, 10)
X_point.shape, Y_point.shape

train: 92 windows x 2000 patches = 184000 examples, X (184000, 113, 1, 1), Y (184000, 6, 2, 10, 1, 1), 0.17 GB


((184000, 113), (184000, 6, 2, 10))

In [8]:
# Patches for training: 64x64 inputs, labels on the central 48x48, ensemble summarized as mean/std
train = make_patch_dataset("train", window_size=64, label_size=48, n_per_window=8, member_mode="meanstd")
train["X"].shape, train["Y"].shape, train["meta"].head(3)

train: 92 windows x 8 patches = 736 examples, X (736, 73, 64, 64), Y (736, 6, 2, 10, 48, 48), 1.69 GB


((736, 73, 64, 64),
 (736, 6, 2, 10, 48, 48),
      month       init  split   i    j
 0  2020-01 2020-01-01  train  87   70
 1  2020-01 2020-01-01  train  75  152
 2  2020-01 2020-01-01  train  77   98)

In [9]:
# Evaluation: tile the full domain (label regions tile exactly with stride = label_size)
val = make_patch_dataset("val", window_size=64, label_size=48, stride=48, member_mode="meanstd")
val["meta"].groupby("init").size().head(3)

val: 24 windows x 20 patches = 480 examples, X (480, 73, 64, 64), Y (480, 6, 2, 10, 48, 48), 1.10 GB


init
2020-01-26 00:00:00    20
2020-01-26 12:00:00    20
2020-01-27 00:00:00    20
dtype: int64

## 7. Sanity checks
1. Re-derive the **linear-interpolation baseline** from a dataset (unnormalized, pointwise, the 5 shared levels).
   It should match `data/data.md` (≈ 0.8 / 1.25 / 1.4 / 1.4 / 1.3 / 1.3 m/s for hours 1–6). This confirms the
   targets line up with the right hours.
2. Normalize → denormalize round trip.

In [10]:
shared = [50, 100, 150, 200, 250]
chk = make_patch_dataset("val", window_size=1, n_per_window=3000, member_mode="mean",
                         target_levels=shared, normalize=False)
names = chk["x_names"]
x0 = np.stack([chk["X"][:, names.index(f"t0_{v}{lev}"), 0, 0] for v in "uv" for lev in shared], 1)
x6 = np.stack([chk["X"][:, names.index(f"t6_mean_{v}{lev}"), 0, 0] for v in "uv" for lev in shared], 1)
y = chk["Y"][..., 0, 0].reshape(len(x0), 6, -1)
rmse = {h: float(np.sqrt(((x0 + (x6 - x0) * h / 6 - y[:, h - 1]) ** 2).mean())) for h in HOURS}
print("linear interp (ERA5 t0 -> FCN3 mean t6) RMSE by hour:", {h: round(r, 3) for h, r in rmse.items()})
assert 0.5 < rmse[1] < 1.1 and 1.0 < rmse[6] < 1.7, "targets look misaligned"

norm = make_patch_dataset("val", window_size=1, n_per_window=3000, member_mode="mean", target_levels=shared)
assert np.allclose(denormalize_y(norm["Y"], shared), chk["Y"], atol=1e-3)
print("round trip OK")

val: 24 windows x 3000 patches = 72000 examples, X (72000, 53, 1, 1), Y (72000, 6, 2, 5, 1, 1), 0.03 GB
linear interp (ERA5 t0 -> FCN3 mean t6) RMSE by hour: {1: 0.765, 2: 1.208, 3: 1.389, 4: 1.378, 5: 1.285, 6: 1.302}


val: 24 windows x 3000 patches = 72000 examples, X (72000, 53, 1, 1), Y (72000, 6, 2, 5, 1, 1), 0.03 GB
round trip OK
